### Create `openalex.works.work_authors` — nightly regeneration (oxjob #684 workstream E)

Replaces `UpdateWorkAuthors`. Design: `work_authors` is a **direct representation of
`openalex_works_base`** (the skeleton: names, strings as displayed, is_corresponding,
raw_orcid, fallbacks) **combined with the author ids already assigned** (`author_id` —
the one path-dependent, non-derivable column). The skeleton is regenerated in full every
run; `author_id` is carried, never recomputed. Staleness of any skeleton field is
impossible by construction (no watermark, no change detection, no drift intake).

**Staging + swap** (no self-referencing reads, no time travel — design rationale and the
research verdict in the #684 job dir, `PLAN-E-work-authors-regeneration.md`):
1. build `work_authors_next` beside the live table (ordinary reads)
2. preservation gate — hard-fail if bound `author_id` count drops beyond threshold
3. guard observation (#608, re-homed) — compares live vs next, telemetry only
4. publish via plain CTAS

**Draft decisions for review (not yet ratified):**
- **Legacy backfill dropped from the carry**: slots new to `work_authors` get
  `author_id = NULL` and flow to MatchAuthors like any unmatched slot (~13K new
  slots/day, essentially all new-era works). The old MERGE's two-pass
  `works_legacy` name-join backfill is retired with it.
- **First run requires `guardrails_override=true`**: regeneration drops the accumulated
  dead-work backlog — ~38.3M phantom rows across ~12.4M works absent from base
  (deleted/merged works the old MERGE never cleaned; grown by the ongoing
  stale-locations cleanup), carrying ~5.6M bound author_ids (measured 2026-08-23).
  After the first run, nightly disappearance tracks cleanup volume and stays under
  the gate threshold.
- `updated_at` is carried but vestigial (the watermark that read it is gone);
  `created_at` carried, stamped fresh for new slots.
- `is_corresponding` is taken verbatim from base (including NULLs) — the old MERGE's
  MAX() was an aggregation artifact.


### Guard prerequisites (oxjob #608) — unchanged

`names_compatible` + the two telemetry tables, idempotent. Enforcement still lives in
MatchAuthors (#649); this notebook only observes.


In [ ]:
CREATE OR REPLACE FUNCTION openalex.authors.names_compatible(
  a_last STRING, a_first STRING, b_last STRING, b_first STRING,
  a_raw STRING, b_raw STRING)
RETURNS BOOLEAN
COMMENT 'oxjob #608 name-compatibility predicate v2. Inputs: author_names.match_last/match_first for both names, plus raw strings for the unparsed fallback. Compatible = same folded surname with initial-or-missing first agreement, OR order-swap, OR surname containment (min length 4). Unparsed side falls back to raw equality.'
RETURN COALESCE(
  CASE
    WHEN a_last IS NULL OR b_last IS NULL
      THEN lower(trim(a_raw)) = lower(trim(b_raw))
    WHEN a_last = b_last
     AND (left(a_first,1) = left(b_first,1) OR a_first IS NULL OR b_first IS NULL)
      THEN TRUE
    WHEN a_last = b_first AND a_first = b_last
      THEN TRUE
    WHEN (startswith(a_last, b_last) OR endswith(a_last, b_last)
          OR startswith(b_last, a_last) OR endswith(b_last, a_last))
     AND least(length(a_last), length(b_last)) >= 4
     AND (left(a_first,1) = left(b_first,1) OR a_first IS NULL OR b_first IS NULL)
      THEN TRUE
    ELSE FALSE
  END, FALSE)

In [ ]:
CREATE TABLE IF NOT EXISTS openalex.authors.author_guard_telemetry (
  run_at TIMESTAMP,
  changed_name_positions BIGINT,
  incompatible BIGINT,
  abstain_unparsed BIGINT,
  abstain_cjk BIGINT,
  curated_holds BIGINT,
  would_invalidate BIGINT,
  rebindable BIGINT,
  realign_tier BIGINT,
  legacy_tier BIGINT,
  isolated_holds BIGINT
)

In [ ]:
CREATE TABLE IF NOT EXISTS openalex.authors.author_guard_events (
  run_at TIMESTAMP,
  work_id BIGINT,
  author_sequence INT,
  incoming_name STRING,
  current_name STRING,
  current_author_id BIGINT,
  verdict STRING,
  work_incompat_count BIGINT,
  curated_hold BOOLEAN,
  invalidate BOOLEAN,
  realign_author_id BIGINT,
  legacy_rebind_id BIGINT,
  rebind_author_id BIGINT,
  postmerge_author_id BIGINT
)

### Step 1: Build `work_authors_next` — the full skeleton with carried state

One statement, ordinary reads of two live tables. Delta CoR is atomic; a failure here
leaves production untouched.


In [ ]:
CREATE OR REPLACE TABLE openalex.works.work_authors_next
CLUSTER BY (work_id) AS
SELECT
    b.work_id,
    b.author_sequence,
    prev.author_id,
    b.raw_author_name,
    b.raw_affiliation_strings,
    b.raw_orcid,
    b.fallback_author_id,
    b.fallback_display_name,
    b.is_corresponding,
    COALESCE(prev.created_at, current_timestamp()) AS created_at,
    COALESCE(prev.updated_at, current_timestamp()) AS updated_at
FROM (
    SELECT id AS work_id,
           pos AS author_sequence,
           au.raw_author_name,
           au.raw_affiliation_strings,
           au.raw_orcid,
           au.author.id AS fallback_author_id,
           au.author.display_name AS fallback_display_name,
           au.is_corresponding
    FROM identifier('openalex' || :env_suffix || '.works.openalex_works_base')
    LATERAL VIEW POSEXPLODE(authorships) t AS pos, au
    WHERE authorships IS NOT NULL AND SIZE(authorships) > 0
) b
LEFT JOIN openalex.works.work_authors prev
    ON b.work_id = prev.work_id AND b.author_sequence = prev.author_sequence

### Step 2: Preservation gate — two checks, two failure modes

`author_id` is path-dependent state with no re-derivation. Two distinct ways to lose it:

1. **Broken carry** (catastrophic): a slot exists in both live and `_next`, live has a
   binding, `_next` came out NULL. Can only mean the carry join or base read is broken.
   **Exact-zero invariant — hard fail, no threshold, no override.**
2. **Mass disappearance** (usually legitimate): works present in live `work_authors` but
   absent from `_next` — works deleted/merged out of base. The stale-locations cleanup
   does this on purpose nightly; a *truncated base read* does it by accident. Thresholded
   and overridable: sized above expected cleanup volume, override for known big waves.

On any failure production is untouched and `work_authors_next` remains for diagnosis.
First run: expect ~12.4M disappeared works (the accumulated dead-work backlog the old
MERGE never cleaned — measured 2026-08-23) — run with `guardrails_override=true`.


In [ ]:
-- Two-part preservation gate. See markdown above.
DECLARE OR REPLACE VARIABLE carry_breaks BIGINT;
DECLARE OR REPLACE VARIABLE disappeared_works BIGINT;
DECLARE OR REPLACE VARIABLE allowed_disappeared BIGINT DEFAULT 2000000;

-- 1. Broken-carry check: bound live slot, present in _next, binding lost
SET VARIABLE carry_breaks = (
  SELECT COUNT(*)
  FROM openalex.works.work_authors live
  JOIN openalex.works.work_authors_next n
    ON live.work_id = n.work_id AND live.author_sequence = n.author_sequence
  WHERE live.author_id IS NOT NULL AND n.author_id IS NULL
);

-- 2. Disappearance check: works that left entirely (deleted/merged works drop here
--    legitimately; a truncated base read drops here catastrophically)
SET VARIABLE disappeared_works = (
  SELECT COUNT(DISTINCT live.work_id)
  FROM openalex.works.work_authors live
  LEFT ANTI JOIN openalex.works.work_authors_next n ON live.work_id = n.work_id
);

SELECT
  carry_breaks AS carry_breaks_must_be_zero,
  disappeared_works,
  allowed_disappeared,
  COALESCE(:guardrails_override, 'false') AS guardrails_override;

SELECT CASE
  WHEN carry_breaks > 0
  THEN RAISE_ERROR(CONCAT(
    'GATE FAILED (carry integrity): ', CAST(carry_breaks AS STRING),
    ' slots present in both tables lost their author_id in work_authors_next. ',
    'The carry join is broken. NOT overridable. Production is untouched; inspect work_authors_next.'))
END;

SELECT CASE
  WHEN disappeared_works > allowed_disappeared
   AND LOWER(COALESCE(:guardrails_override, 'false')) <> 'true'
  THEN RAISE_ERROR(CONCAT(
    'GATE FAILED (mass disappearance): ', CAST(disappeared_works AS STRING),
    ' works present in work_authors are absent from work_authors_next (allowed: ',
    CAST(allowed_disappeared AS STRING),
    '). If a known cleanup/merge wave, set guardrails_override=true; otherwise the base read is suspect. ',
    'Production is untouched; inspect work_authors_next.'))
END;

### Step 3: Name-transition guard — OBSERVATION ONLY (oxjob #608), re-homed

Same verdicts, corroboration, and hypothetical-rebind cascade as before; the only change
is admission. The old batch compared watermark-selected incoming names against live
seats; this compares `work_authors_next` (what tonight will publish) against the live
table (what yesterday published) — every name transition in the corpus is seen, not
just watermark-touched works. Runs BEFORE publish so both states exist.
Nothing here changes any binding — enforcement lives in MatchAuthors (#649).


In [ ]:
CREATE OR REPLACE TABLE openalex.authors.work_authors_guard_batch AS
WITH changed AS (
    -- admission: every live bound seat whose incoming (next) name differs
    SELECT DISTINCT
        n.work_id,
        n.author_sequence,
        n.raw_author_name AS incoming_name,
        ws.raw_author_name AS current_name,
        ws.author_id AS current_author_id
    FROM openalex.works.work_authors_next n
    JOIN openalex.works.work_authors ws
        ON n.work_id = ws.work_id AND n.author_sequence = ws.author_sequence
    WHERE ws.author_id IS NOT NULL
      -- null-safe: a name appearing or disappearing is a judged transition too
      AND NOT (LOWER(TRIM(n.raw_author_name)) <=> LOWER(TRIM(ws.raw_author_name)))
),
work_seats AS (
    -- every bound seat of the works admitted above (reused by the rebind cascade)
    SELECT wa.work_id, wa.author_sequence, wa.raw_author_name, wa.author_id
    FROM openalex.works.work_authors wa
    JOIN (SELECT DISTINCT work_id FROM changed) cw ON wa.work_id = cw.work_id
    WHERE wa.author_id IS NOT NULL
),
judged AS (
    SELECT c.*,
        an_i.match_last AS in_last, an_i.match_first AS in_first,
        an_c.match_last AS cur_last, an_c.match_first AS cur_first,
        -- Three-state verdict. ABSTAIN classes neither act nor corroborate: the raw-equality
        -- fallback in names_compatible is unreachable here (admission requires differing raws),
        -- so unparsed names CANNOT be judged; CJK is the frozen-parser false-positive class.
        CASE
            WHEN c.incoming_name RLIKE '[\u1100-\u11FF\u3040-\u30FF\u3130-\u318F\u3400-\u4DBF\u4E00-\u9FFF\uAC00-\uD7AF\uF900-\uFAFF]'
              OR c.current_name RLIKE '[\u1100-\u11FF\u3040-\u30FF\u3130-\u318F\u3400-\u4DBF\u4E00-\u9FFF\uAC00-\uD7AF\uF900-\uFAFF]' THEN 'ABSTAIN_CJK'
            WHEN an_i.match_last IS NULL OR an_c.match_last IS NULL THEN 'ABSTAIN_UNPARSED'
            WHEN openalex.authors.names_compatible(
                     an_i.match_last, an_i.match_first,
                     an_c.match_last, an_c.match_first,
                     c.incoming_name, c.current_name) THEN 'COMPATIBLE'
            ELSE 'INCOMPATIBLE'
        END AS verdict
    FROM changed c
    LEFT JOIN openalex.authors.author_names an_i ON TRIM(c.incoming_name) = an_i.raw_author_name
    LEFT JOIN openalex.authors.author_names an_c ON TRIM(c.current_name) = an_c.raw_author_name
),
counted AS (
    SELECT *,
        COUNT(CASE WHEN verdict = 'INCOMPATIBLE' THEN 1 END) OVER (PARTITION BY work_id) AS work_incompat_count
    FROM judged
),
curated AS (
    SELECT DISTINCT c.work_id, c.author_sequence
    FROM counted c
    JOIN openalex.works.work_author_claim_curations cc
        ON cc.work_id = c.work_id
       AND LOWER(TRIM(cc.raw_author_name)) = LOWER(TRIM(c.incoming_name))
),
flagged AS (
    SELECT c.*,
        (cu.work_id IS NOT NULL) AS curated_hold,
        (c.verdict = 'INCOMPATIBLE' AND c.work_incompat_count >= 2
         AND cu.work_id IS NULL) AS invalidate
    FROM counted c
    LEFT JOIN curated cu ON c.work_id = cu.work_id AND c.author_sequence = cu.author_sequence
),
-- Hypothetical rebind cascade (OBSERVATION ONLY — a hypothesis against THIS RUN's pre-publish
-- state, recorded to events; any consumer MUST revalidate against live state at apply time).
freed_pairs AS (
    SELECT f.work_id, f.current_name AS donor_name, f.current_author_id AS donor_id,
           f.cur_last AS donor_last, f.cur_first AS donor_first
    FROM flagged f
    LEFT JOIN openalex.authors.openalex_authors oa ON f.current_author_id = oa.id
    LEFT JOIN openalex.authors.authors ar ON f.current_author_id = ar.id
    LEFT JOIN openalex.authors.author_names an_p
        ON TRIM(COALESCE(oa.display_name, ar.display_name)) = an_p.raw_author_name
    -- curated display_names can be absent from author_names; fall back to full_name keys
    LEFT JOIN openalex.authors.author_names an_pf
        ON TRIM(oa.full_name) = an_pf.raw_author_name
    WHERE f.invalidate
      AND openalex.authors.names_compatible(
            f.cur_last, f.cur_first,
            CASE WHEN an_p.match_last IS NOT NULL THEN an_p.match_last ELSE an_pf.match_last END,
            CASE WHEN an_p.match_last IS NOT NULL THEN an_p.match_first ELSE an_pf.match_first END,
            f.current_name, COALESCE(oa.display_name, ar.display_name))
),
realign_cand AS (
    SELECT f.work_id, f.author_sequence,
        COUNT(DISTINCT CASE WHEN LOWER(TRIM(fp.donor_name)) = LOWER(TRIM(f.incoming_name))
                            THEN fp.donor_id END) AS n_exact,
        MIN(CASE WHEN LOWER(TRIM(fp.donor_name)) = LOWER(TRIM(f.incoming_name))
                 THEN fp.donor_id END) AS id_exact,
        COUNT(DISTINCT CASE WHEN openalex.authors.names_compatible(
                                 f.in_last, f.in_first, fp.donor_last, fp.donor_first,
                                 f.incoming_name, fp.donor_name)
                            THEN fp.donor_id END) AS n_compat,
        MIN(CASE WHEN openalex.authors.names_compatible(
                      f.in_last, f.in_first, fp.donor_last, fp.donor_first,
                      f.incoming_name, fp.donor_name)
                 THEN fp.donor_id END) AS id_compat
    FROM flagged f
    JOIN freed_pairs fp ON fp.work_id = f.work_id
    WHERE f.invalidate
    GROUP BY f.work_id, f.author_sequence
),
realign_chosen AS (
    SELECT work_id, author_sequence,
        CASE WHEN n_exact = 1 THEN id_exact
             WHEN n_exact = 0 AND n_compat = 1 THEN id_compat
        END AS chosen_id
    FROM realign_cand
),
realign_unique AS (
    SELECT work_id, author_sequence, chosen_id,
        COUNT(*) OVER (PARTITION BY work_id, chosen_id) AS n_receivers
    FROM realign_chosen
    WHERE chosen_id IS NOT NULL
),
legacy_ok AS (
    SELECT DISTINCT f.work_id, f.author_sequence
    FROM flagged f
    JOIN openalex.works_legacy.work_authors l
        ON l.work_id = f.work_id AND l.author_id = f.current_author_id
    WHERE f.invalidate
),
-- NOTE: stricter than the retired MERGE's rank-1 legacy resolution ON PURPOSE — observation
-- reports honest uniqueness (0-or-2+ candidates -> no hypothesis) rather than an arbitrary pick.
legacy_exact AS (
    SELECT f.work_id, f.author_sequence,
           MIN(l.author_id) AS id_l, COUNT(DISTINCT l.author_id) AS n_l
    FROM flagged f
    JOIN openalex.works_legacy.work_authors l
        ON l.work_id = f.work_id
       AND LOWER(TRIM(l.raw_author_name)) = LOWER(TRIM(f.incoming_name))
    WHERE f.invalidate AND l.author_id IS NOT NULL
    GROUP BY f.work_id, f.author_sequence
),
legacy_parsed AS (
    SELECT f.work_id, f.author_sequence,
           MIN(l.author_id) AS id_l, COUNT(DISTINCT l.author_id) AS n_l
    FROM flagged f
    JOIN openalex.works_legacy.work_authors l
        ON l.work_id = f.work_id AND l.author_id IS NOT NULL
    JOIN openalex.authors.author_names pn ON TRIM(l.raw_author_name) = pn.raw_author_name
    WHERE f.invalidate AND f.in_last IS NOT NULL
      AND pn.match_last = f.in_last
      AND COALESCE(pn.match_first, '') = COALESCE(f.in_first, '')
    GROUP BY f.work_id, f.author_sequence
),
with_rebind AS (
    SELECT f.*,
        CASE WHEN r.n_receivers = 1 THEN r.chosen_id END AS realign_author_id,
        (lk.work_id IS NOT NULL) AS legacy_fallback_ok,
        CASE WHEN le.n_l = 1 THEN le.id_l
             WHEN le.work_id IS NULL AND lp.n_l = 1 THEN lp.id_l
        END AS legacy_rebind_id
    FROM flagged f
    LEFT JOIN realign_unique r ON f.work_id = r.work_id AND f.author_sequence = r.author_sequence
    LEFT JOIN legacy_ok lk ON f.work_id = lk.work_id AND f.author_sequence = lk.author_sequence
    LEFT JOIN legacy_exact le ON f.work_id = le.work_id AND f.author_sequence = le.author_sequence
    LEFT JOIN legacy_parsed lp ON f.work_id = lp.work_id AND f.author_sequence = lp.author_sequence
),
candidate AS (
    SELECT *,
        COALESCE(realign_author_id,
                 CASE WHEN legacy_fallback_ok THEN legacy_rebind_id END) AS rebind_candidate
    FROM with_rebind
),
occupied AS (
    SELECT DISTINCT ws.work_id, ws.author_id
    FROM work_seats ws
    LEFT ANTI JOIN (SELECT work_id, author_sequence FROM flagged WHERE invalidate) inv
        ON ws.work_id = inv.work_id AND ws.author_sequence = inv.author_sequence
),
-- an author a curator explicitly removed from the work is never a valid hypothesis
removes AS (
    SELECT DISTINCT rc.work_id, rc.author_id
    FROM openalex.works.work_author_remove_curations rc
    JOIN (SELECT DISTINCT work_id FROM changed) cw ON rc.work_id = cw.work_id
)
SELECT c.*,
    CASE WHEN c.rebind_candidate IS NULL THEN NULL
         WHEN o.author_id IS NOT NULL THEN NULL
         WHEN rm.author_id IS NOT NULL THEN NULL
         WHEN COUNT(*) OVER (PARTITION BY c.work_id, c.rebind_candidate) > 1 THEN NULL
         ELSE c.rebind_candidate
    END AS rebind_author_id
FROM candidate c
LEFT JOIN occupied o ON c.work_id = o.work_id AND c.rebind_candidate = o.author_id
LEFT JOIN removes rm ON c.work_id = rm.work_id AND c.rebind_candidate = rm.author_id

In [ ]:
INSERT INTO openalex.authors.author_guard_events
SELECT current_timestamp(), gb.work_id, gb.author_sequence,
       gb.incoming_name, gb.current_name, gb.current_author_id,
       gb.verdict, gb.work_incompat_count, gb.curated_hold, gb.invalidate,
       gb.realign_author_id, gb.legacy_rebind_id, gb.rebind_author_id,
       -- post-publish binding = the carried author_id in _next (what tonight publishes)
       n.author_id AS postmerge_author_id
FROM openalex.authors.work_authors_guard_batch gb
LEFT JOIN openalex.works.work_authors_next n
    ON n.work_id = gb.work_id AND n.author_sequence = gb.author_sequence
WHERE gb.verdict != 'COMPATIBLE'
  -- prod-only: dev-suffix runs must not contaminate the append-only prod log
  AND :env_suffix = ''

In [ ]:
INSERT INTO openalex.authors.author_guard_telemetry
SELECT
    current_timestamp() AS run_at,
    COUNT(*) AS changed_name_positions,
    COUNT(CASE WHEN verdict = 'INCOMPATIBLE' THEN 1 END) AS incompatible,
    COUNT(CASE WHEN verdict = 'ABSTAIN_UNPARSED' THEN 1 END) AS abstain_unparsed,
    COUNT(CASE WHEN verdict = 'ABSTAIN_CJK' THEN 1 END) AS abstain_cjk,
    COUNT(CASE WHEN curated_hold AND verdict = 'INCOMPATIBLE' THEN 1 END) AS curated_holds,
    COUNT(CASE WHEN invalidate THEN 1 END) AS would_invalidate,
    COUNT(CASE WHEN invalidate AND rebind_author_id IS NOT NULL THEN 1 END) AS rebindable,
    COUNT(CASE WHEN invalidate AND rebind_author_id IS NOT NULL
                AND rebind_author_id = realign_author_id THEN 1 END) AS realign_tier,
    COUNT(CASE WHEN invalidate AND rebind_author_id IS NOT NULL
                AND (realign_author_id IS NULL OR rebind_author_id != realign_author_id) THEN 1 END) AS legacy_tier,
    -- disjoint from curated_holds: incompatible = would_invalidate + curated_holds + isolated_holds
    COUNT(CASE WHEN verdict = 'INCOMPATIBLE' AND work_incompat_count = 1
                AND NOT curated_hold THEN 1 END) AS isolated_holds
FROM openalex.authors.work_authors_guard_batch
HAVING :env_suffix = ''

### Step 4: Publish

Atomic swap for readers; `work_authors_next` is left in place between runs as a free
diagnosis artifact.


In [ ]:
CREATE OR REPLACE TABLE openalex.works.work_authors
CLUSTER BY (work_id) AS
SELECT * FROM openalex.works.work_authors_next

### Verify


In [ ]:
SELECT
  COUNT(*) AS rows,
  COUNT(author_id) AS bound,
  COUNT(*) - COUNT(author_id) AS unbound,
  MAX(created_at) AS newest_slot
FROM openalex.works.work_authors